In [1]:
import pandas as pd


In [2]:
df_1 = pd.read_csv("../../../sales.csv")
df_1


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79
...,...,...,...
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15


In [3]:
df_2 = pd.read_csv("../../../sample_submission.csv")
df_2


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24
...,...,...,...
543,2024-06-27,4111585.95,4048492.76
544,2024-06-28,4591751.39,4504603.05
545,2024-06-29,6582807.93,6478608.03
546,2024-06-30,5484749.90,5386065.13


In [4]:
df = pd.concat([df_1, df_2], ignore_index=True)
df


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79
...,...,...,...
4376,2024-06-27,4111585.95,4048492.76
4377,2024-06-28,4591751.39,4504603.05
4378,2024-06-29,6582807.93,6478608.03
4379,2024-06-30,5484749.90,5386065.13


In [5]:
df['Date'] = pd.to_datetime(df['Date'])
# chuyển sang day of year
df['doy'] = df['Date'].dt.dayofyear

# define promo ranges (MM-DD)
promo_ranges = [
    ("03-18", "04-17"),
    ("06-23", "07-22"),
    ("11-18", "01-02"),  # qua năm
    ("08-30", "10-02")
]


In [6]:
def mmdd_to_doy(mmdd):
    return pd.to_datetime(f"2021-{mmdd}").dayofyear  # năm bất kỳ không nhuận


In [7]:
# convert range
promo_doy_ranges = [(mmdd_to_doy(s), mmdd_to_doy(e)) for s, e in promo_ranges]


In [8]:
def is_promo(doy):
    for start, end in promo_doy_ranges:
        if start <= end:
            if start <= doy <= end:
                return 1
        else:
            # case qua năm (wrap)
            if doy >= start or doy <= end:
                return 1
    return 0


In [9]:
df['is_promo'] = df['doy'].apply(is_promo)
df['is_promo'].value_counts()



is_promo
0    2688
1    1693
Name: count, dtype: int64

In [10]:
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month


In [11]:
df


,Date,Revenue,COGS,doy,is_promo,year,month
0,2012-07-04,5123547.94,3982991.19,186,1,2012,7
1,2012-07-05,2751773.45,2150580.23,187,1,2012,7
2,2012-07-06,3054029.42,2517632.84,188,1,2012,7
3,2012-07-07,2667930.94,2108246.62,189,1,2012,7
4,2012-07-08,2360851.90,1808622.79,190,1,2012,7
...,...,...,...,...,...,...,...
4376,2024-06-27,4111585.95,4048492.76,179,1,2024,6
4377,2024-06-28,4591751.39,4504603.05,180,1,2024,6
4378,2024-06-29,6582807.93,6478608.03,181,1,2024,6
4379,2024-06-30,5484749.90,5386065.13,182,1,2024,6


In [12]:
df['quarter'] = df['Date'].dt.quarter
df


,Date,Revenue,COGS,doy,is_promo,year,month,quarter
0,2012-07-04,5123547.94,3982991.19,186,1,2012,7,3
1,2012-07-05,2751773.45,2150580.23,187,1,2012,7,3
2,2012-07-06,3054029.42,2517632.84,188,1,2012,7,3
3,2012-07-07,2667930.94,2108246.62,189,1,2012,7,3
4,2012-07-08,2360851.90,1808622.79,190,1,2012,7,3
...,...,...,...,...,...,...,...,...
4376,2024-06-27,4111585.95,4048492.76,179,1,2024,6,2
4377,2024-06-28,4591751.39,4504603.05,180,1,2024,6,2
4378,2024-06-29,6582807.93,6478608.03,181,1,2024,6,2
4379,2024-06-30,5484749.90,5386065.13,182,1,2024,6,2


In [13]:
def get_season(month):
    if month in [2, 3, 4]:
        return 'Spring'
    elif month in [5, 6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return 'Winter'  # 12, 1

df['season'] = df['month'].apply(get_season)


In [14]:
import numpy as np
df['days_in_year'] = df['Date'].dt.is_leap_year.apply(lambda x: 366 if x else 365)

df['doy_sin'] = np.sin(2 * np.pi * df['doy'] / df['days_in_year'])
df['doy_cos'] = np.cos(2 * np.pi * df['doy'] / df['days_in_year'])


In [15]:
df['promo_past_14'] = df['is_promo'].rolling(14, min_periods=1).sum()
df['promo_past_28'] = df['is_promo'].rolling(28, min_periods=1).sum()
df['promo_past_42'] = df['is_promo'].rolling(42, min_periods=1).sum()
df.tail(10)


,Date,Revenue,COGS,doy,is_promo,year,month,quarter,season,days_in_year,doy_sin,doy_cos,promo_past_14,promo_past_28,promo_past_42
4371,2024-06-22,4333295.02,3481587.38,174,1,2024,6,2,Summer,366,1.538906e-01,-0.988088,1.0,1.0,1.0
4372,2024-06-23,4131238.18,3416011.51,175,1,2024,6,2,Summer,366,1.369061e-01,-0.990584,2.0,2.0,2.0
4373,2024-06-24,4356931.96,3802125.11,176,1,2024,6,2,Summer,366,1.198812e-01,-0.992788,3.0,3.0,3.0
4374,2024-06-25,4183743.82,3792016.38,177,1,2024,6,2,Summer,366,1.028210e-01,-0.994700,4.0,4.0,4.0
4375,2024-06-26,4201484.02,3925647.48,178,1,2024,6,2,Summer,366,8.573050e-02,-0.996318,5.0,5.0,5.0
4376,2024-06-27,4111585.95,4048492.76,179,1,2024,6,2,Summer,366,6.861474e-02,-0.997643,6.0,6.0,6.0
4377,2024-06-28,4591751.39,4504603.05,180,1,2024,6,2,Summer,366,5.147875e-02,-0.998674,7.0,7.0,7.0
4378,2024-06-29,6582807.93,6478608.03,181,1,2024,6,2,Summer,366,3.432760e-02,-0.999411,8.0,8.0,8.0
4379,2024-06-30,5484749.90,5386065.13,182,1,2024,6,2,Summer,366,1.716633e-02,-0.999853,9.0,9.0,9.0
4380,2024-07-01,5057610.99,4989753.02,183,1,2024,7,3,Summer,366,1.224647e-16,-1.000000,10.0,10.0,10.0


In [16]:
df_test = df[df['Date'] >= pd.to_datetime("2023-01-01")]
df_test.drop(columns=(['Revenue','COGS']), inplace=True)
df_test.to_csv("test_prepared_2.csv", index=False)


/var/folders/36/khn_y8zn7qqflczfpst8cbfc0000gn/T/ipykernel_49328/3914750459.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=(['Revenue','COGS']), inplace=True)


In [17]:
df_test


,Date,doy,is_promo,year,month,quarter,season,days_in_year,doy_sin,doy_cos,promo_past_14,promo_past_28,promo_past_42
3833,2023-01-01,1,1,2023,1,1,Winter,365,1.721336e-02,0.999852,14.0,28.0,42.0
3834,2023-01-02,2,1,2023,1,1,Winter,365,3.442161e-02,0.999407,14.0,28.0,42.0
3835,2023-01-03,3,0,2023,1,1,Winter,365,5.161967e-02,0.998667,13.0,27.0,41.0
3836,2023-01-04,4,0,2023,1,1,Winter,365,6.880243e-02,0.997630,12.0,26.0,40.0
3837,2023-01-05,5,0,2023,1,1,Winter,365,8.596480e-02,0.996298,11.0,25.0,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4376,2024-06-27,179,1,2024,6,2,Summer,366,6.861474e-02,-0.997643,6.0,6.0,6.0
4377,2024-06-28,180,1,2024,6,2,Summer,366,5.147875e-02,-0.998674,7.0,7.0,7.0
4378,2024-06-29,181,1,2024,6,2,Summer,366,3.432760e-02,-0.999411,8.0,8.0,8.0
4379,2024-06-30,182,1,2024,6,2,Summer,366,1.716633e-02,-0.999853,9.0,9.0,9.0


In [18]:
import joblib
cogs_model = joblib.load('../cogs_model.pkl')
rev_model = joblib.load('../rev_model.pkl')


In [19]:
ans = pd.read_csv('../../../sample_submission.csv')
ans


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24
...,...,...,...
543,2024-06-27,4111585.95,4048492.76
544,2024-06-28,4591751.39,4504603.05
545,2024-06-29,6582807.93,6478608.03
546,2024-06-30,5484749.90,5386065.13


In [20]:
dict_season = {'Autumn': 0, 'Spring': 1, 'Summer': 2, 'Winter': 3}
df_test['season'] = df_test['season'].map(dict_season)


/var/folders/36/khn_y8zn7qqflczfpst8cbfc0000gn/T/ipykernel_49328/1200536486.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['season'] = df_test['season'].map(dict_season)


In [21]:
ans['Revenue'] = rev_model.predict(df_test.drop(columns=['Date']))
ans['COGS'] = cogs_model.predict(df_test.drop(columns=['Date']))


In [22]:
ans


,Date,Revenue,COGS
0,2023-01-01,2.183031e+06,2.497797e+06
1,2023-01-02,1.722752e+06,1.631925e+06
2,2023-01-03,1.064885e+06,7.484007e+05
3,2023-01-04,1.064885e+06,9.287551e+05
4,2023-01-05,1.217244e+06,7.944149e+05
...,...,...,...
543,2024-06-27,3.636322e+06,3.933124e+06
544,2024-06-28,4.346845e+06,5.768921e+06
545,2024-06-29,5.597314e+06,5.599498e+06
546,2024-06-30,5.363479e+06,5.648775e+06


In [23]:
ans.to_csv("submission.csv", index=False)
